# Launching ND AI Lab

AI lab can also be launched with a pixi command, from the `pixi` folder of the repo:

```sh
cd i2k-2026/pixi
pixi run bees
```

`pixi run pollen` and `pixi run ladybugs` open the other two datasets.

Or as a Napari plugin - `Plugins > Napari AI Lab >` then pick one of the two profiles:

- **ND AI Lab** - everything installed
- **2D Instance AI Lab** - only the scikit-ops segmenters, and a simpler GUI

It's convenient to launch from a notebook though because we can setup everything as we want it. 

For example set `DATASET` below to choose the project. 

## Why AI Lab

- The built-in models do not find the bees well enough.
- To do better we train a model.
- Training needs labels.
- Labels are drawn by a person, looking at the images.
- AI Lab is where we draw the labels.
- AI lab can also do the other parts (segment with builtins, augment, train)
- other parts can be done with AI-lab or scripts, labeling only with AI-lab because need a GUI

What AI Lab does:

- Allows you to work with folders of images as a sequence
- Can use Napari's built in labeling functionality to draw the objects
- Can optionally label with SAM 
    - Click on an object and SAM draws the outline.
- Draw a label box around the part you are finished with. Only that part is used for training.
- Run Cellpose, copy its predictions into labels, then fix the mistakes.
- Augment, train and predict from buttons.
- Step through all the images in a set.

## Choose the project

| dataset | what it is |
|---|---|
| `bees` | seven comb images, already labelled |
| `ladybugs` | eleven photographs, no annotations yet |
| `pollen_count` | ten slides, already labelled |

`ladybugs` starts empty, so ND AI Lab creates the annotation folders on
first save. The other two open with their existing labels.

In [1]:
DATASET = 'bees'   # 'bees' | 'ladybugs' | 'pollen_count'

from pathlib import Path

# Works whether the kernel starts in notebooks/ or at the repo root.
for candidate in (Path.cwd() / "data", Path.cwd() / "notebooks" / "data"):
    if (candidate / DATASET).is_dir():
        project = candidate / DATASET
        break
else:
    raise FileNotFoundError(f"{DATASET} not found from {Path.cwd()}")

images = sorted(project.glob('*.png')) + sorted(project.glob('*.jpg'))
print(project)
print(len(images), 'images')

c:\Users\bnort\work\ImageJ2022\tnia\i2k-2026\notebooks\data\bees
7 images


## Launch

`register_all=True` registers every segmenter and augmenter, so the
dropdowns are populated. Pass `False` to register only what you import
yourself.

In [2]:
import napari
from napari_ai_lab.apps.nd_ai_lab_launcher import launch_nd_ai_lab

from napari_ai_lab.apps.profiles import list_profiles, get_profile

list_profiles()

['2d-instance-skop', 'all']

In [3]:
viewer = napari.Viewer()

ai_lab, sequence_viewer, model = launch_nd_ai_lab(
    viewer,
    project,
    viewer_type="sequence",
    axes_to_collapse="C",
    axis_types="NYXC",
    register_all=True,
    profile="2d-instance-skop",
)

print('ND AI Lab launched on', project.name)

INFO:OpenGL.acceleratesupport:No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


Registering profile: 2d-instance-skop
Registered global segmenter: StarDist2D (scikit-ops)
Registered global segmenter: Cellpose3 (scikit-ops)
Registered global segmenter: Cellpose4 (scikit-ops)
Registered interactive segmenter: Otsu2D
Registered interactive segmenter: SAM3D
Registered augmenter: SimpleAugmenter
Registered augmenter: AlbumentationsAugmenter
Connected to viewer close event
Potential axes: ['YX']
Supported axes (before filtering): ['YX', 'YXC']
Added instructions for Otsu2D
Added axis selection for Otsu2D: ['YX', 'YXC']
Selected segmenter: Otsu2D
Supported axes: ['YX', 'YXC']
Potential axes: ['YX']
Supported axes (before filtering): ['YX', 'YXC']
Added instructions for Otsu2D
Added axis selection for Otsu2D: ['YX', 'YXC']
Selected segmenter: Otsu2D
Supported axes: ['YX', 'YXC']
Viewer close event handler already installed - skipping
Selected augmenter: AlbumentationsAugmenter
Created new augmenter instance: AlbumentationsAugmenter
Augmenter type: <class 'napari_ai_lab.Au

C:\Users\bnort\work\ImageJ2022\tnia\i2k-2026\pixi\.pixi\envs\default\Lib\site-packages\napari\layers\utils\style_encoding.py:251: RuntimeWarning: Applying the encoding failed. Using the safe fallback value instead.
  warnings.warn(


Viewer closing detected via closeEvent


## Why those arguments

| argument | why |
|---|---|
| `viewer_type="sequence"` | all three sets hold images of different shapes, so they cannot be one stacked array |
| `axes_to_collapse="C"` | colour is not an axis to annotate along; labels are 2D per image |
| `axis_types="NYXC"` | N images, each Y by X with a colour axis |

The same three values suit all three datasets. A project of equal-sized
images could use `viewer_type="stacked"` instead.

## Using it

The **Sequence Viewer** at the bottom moves between images. The **AI Lab**
dock on the right has Label, Augment and Segment.

Interactive segmentation lives on the Label tab. SAM3D there needs
micro_sam, which the pixi environment has and the pip fallback does not.